In [ ]:
import pandas as pd

df = pd.read_csv("HT.csv")

In [ ]:
df.head(5)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().mean() * 100

In [ ]:
df["session_duration"] = df["session_duration"].fillna(
    df["session_duration"].median()
)

df["satisfaction"] = df["satisfaction"].fillna(
    df["satisfaction"].median()
)

In [ ]:
df["country"] = df["country"].fillna("Unknown")

In [ ]:
df.isnull().sum()

**Standard deviation tells us how much the data values are spread out around the mean. Age standard deviation (8.48) means the ages typically vary about 8.48 years from the average age.**

In [ ]:
df.describe()

The 25%, 50%, and 75% values in describe() are percentiles.

Simply:

* 25% → 25% of the data is below or equal to this value.
* 50% → 50% of the data is below or equal to this value → this is called the median.
* 75% → 75% of the data is below or equal to this value.


* **25% = 25** → 25% of the people are **25 years old or younger**.
* **50% = 31** → 50% of the people are **31 years old or younger**.
* **75% = 37** → 75% of the people are **37 years old or younger**.


In [ ]:
df["purchased"].value_counts()

In [ ]:
df["group"].value_counts() # Control = old version, Treatment = New version


In [ ]:
df["group"].value_counts(normalize=True) * 100

In [ ]:
df.groupby("group")["purchased"].agg(["count", "sum", "mean"])

In [ ]:
df.groupby("group")["purchased"].mean() # 0 = did not purchase
#1 = purchased

Control conversion = 20%
Treatment conversion = 23.1%

**Business Question

Does the new version increase the purchase conversion rate compared with the old version?**

Null Hypothesis — H₀

The treatment group has the same purchase conversion rate as the control group.

Mathematically:

$$ H_0: p_T = p_C $$
Alternative Hypothesis — H₁

H1​:pT​>pC​

H₀: Treatment conversion = Control conversion

H₁: Treatment conversion > Control conversion

In [ ]:
control = df[df["group"] == "Control"]
treatment = df[df["group"] == "Treatment"]

In [ ]:
print("Control users:", len(control))
print("Treatment users:", len(treatment))

In [ ]:
control_purchases = control["purchased"].sum()
treatment_purchases = treatment["purchased"].sum()

print("Control purchases:", control_purchases)
print("Treatment purchases:", treatment_purchases)

In [ ]:
control_users = len(control)
treatment_users = len(treatment)

print("Control users:", control_users)
print("Treatment users:", treatment_users)

In [ ]:
count = [
    treatment_purchases,
    control_purchases
]

nobs = [
    treatment_users,
    control_users
]

In [ ]:
print("count:", count)
print("nobs:", nobs)

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

z_stat, p_value = proportions_ztest(
    count,
    nobs,
    alternative="larger"
)

print("Z-statistic:", z_stat)
print("P-value:", p_value)

In [ ]:
alpha = 0.05

if p_value < alpha:
    print("Reject H0")
else:
    print("Fail to reject H0")

**Absolute Difference / Lift**

In [ ]:
control_rate = control["purchased"].mean()
treatment_rate = treatment["purchased"].mean()

absolute_lift = treatment_rate - control_rate

print("Control conversion:", control_rate)
print("Treatment conversion:", treatment_rate)
print("Absolute lift:", absolute_lift)
print("Absolute lift (% points):", absolute_lift * 100)

**Relative Lift**

In [ ]:
relative_lift = (
    (treatment_rate - control_rate)
    / control_rate
)

print("Relative Lift:", relative_lift)
print("Relative Lift (%):", relative_lift * 100)

**Treatment increased conversion by approximately 15% relative to the control group**

**Treatment-এর actual conversion improvement কত হতে পারে?**

In [ ]:
from statsmodels.stats.proportion import confint_proportions_2indep

In [ ]:
ci_low, ci_high = confint_proportions_2indep(
    count1=treatment_purchases,
    nobs1=treatment_users,
    count2=control_purchases,
    nobs2=control_users,
    method="wald"
)

print("95% Confidence Interval")
print("Lower:", ci_low)
print("Upper:", ci_high)

In [ ]:
print("Lower:", ci_low * 100, "percentage points")
print("Upper:", ci_high * 100, "percentage points")

**The treatment improved conversion by approximately 3.02 percentage points, with a 95% confidence interval indicating that the true improvement is likely between approximately 1.8 and 4.2 percentage points.**

**A/B Test Conclusion**

The treatment group achieved a higher purchase conversion rate than the control group. The control group had a conversion rate of approximately 20.09%, while the treatment group achieved approximately 23.11%, resulting in an absolute improvement of about 3.02 percentage points.

A one-sided two-proportion z-test was conducted to determine whether the treatment conversion rate was significantly higher than the control conversion rate. The resulting p-value was far below the 0.05 significance level, so we reject the null hypothesis.

This provides strong statistical evidence that the new version improves purchase conversion compared with the old version.

The next step is to evaluate the confidence interval and practical business impact before making a final rollout decision.